# Text-to-Speech with VITS and ArTST

Run all cells in a fresh Python 3.11 kernel. Both complete original paragraphs are preserved. All checkpoints and the speaker dataset are revision-pinned. English VITS uses bundled eSpeak; Arabic ArTST uses an explicit pinned SpeechT5 vocoder. ArTST model license: CC-BY-NC 4.0; this is an educational noncommercial example, not a commercial deployment grant. Actual waveform checks do not establish pronunciation quality. CPU execution, no API keys. Originals remain nonexecuting below.

In [ ]:
import os,sys,subprocess
from pathlib import Path
from tempfile import TemporaryDirectory
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','torch==2.10.0','transformers==4.57.6','datasets==3.6.0','soundfile==0.13.1','sentencepiece==0.2.1','phonemizer==3.3.0','espeakng-loader==0.2.4','inflect==7.5.0','numpy==2.3.5','certifi==2026.10.23'],check=True)
os.environ['USE_TF']='0'
os.environ['USE_FLAX']='0'
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS']='1'
import certifi
os.environ['SSL_CERT_FILE']=certifi.where()
os.environ['REQUESTS_CA_BUNDLE']=certifi.where()
import numpy as np
import torch
import soundfile as sf
import espeakng_loader
from phonemizer.backend.espeak.wrapper import EspeakWrapper
os.environ['ESPEAK_DATA_PATH']=espeakng_loader.get_data_path()
EspeakWrapper.set_library(espeakng_loader.get_library_path())
torch.set_num_threads(2)
workspace=TemporaryDirectory(prefix='tdb-vits-artst-')
work_dir=Path(workspace.name)
VITS_ID='kakao-enterprise/vits-ljs'
VITS_REV='3bcb8321394f671bd948ebf0d086d694dda95464'
ARTST_ID='MBZUAI/speecht5_tts_clartts_ar'
ARTST_REV='7343bf848826aa10e6b349ee92f91867b06178b7'
VOCODER_REV='bb6f429406e86a9992357a972c0698b22043307d'
EMBEDDING_REV='3662f5496eed3bcee8f1fcaaad028a1b99eccff6'
print('Python:',sys.version.split()[0],'| PyTorch:',torch.__version__,'| device: cpu')
print('Pinned VITS, ArTST, vocoder and speaker dataset; bundled eSpeak phonemizer ready.')

In [ ]:
print('Pinned Transformers, datasets, soundfile, phonemizer and bundled eSpeak dependencies installed in setup.')

In [ ]:
from transformers import pipeline, SpeechT5HifiGan
from datasets import load_dataset
from IPython.display import Audio as IPythonAudio, display
print('Text-to-speech, dataset, audio playback and WAV imports ready.')

In [ ]:
narrator=pipeline('text-to-speech',model=VITS_ID,revision=VITS_REV,device='cpu')
print('English narrator:',VITS_ID,'| sample rate:',narrator.model.config.sampling_rate)
text = '\nThe Israeli occupation of Palestine began in 1967 \nduring the Six-Day War when Israel captured the West Bank, \nGaza Strip, and East Jerusalem. \nThese areas, home to many Palestinians, have since been a \nfocal point of conflict. The international community generally views Israeli settlements there as illegal. \nEfforts towards peace continue, with Palestinians seeking \nindependence and Israelis seeking security. \nThe situation remains highly complex and contentious.\n'
english_text=text
torch.manual_seed(42)
narrated_text=narrator(text)
english_audio=np.asarray(narrated_text['audio']).reshape(-1)
english_rate=narrated_text['sampling_rate']
display(IPythonAudio(english_audio,rate=english_rate))
print('English waveform:',english_audio.shape,'| sample rate:',english_rate,'| duration:',round(len(english_audio)/english_rate,3),'seconds')

In [ ]:
display(IPythonAudio(english_audio,rate=english_rate))
print('English audio player uses the complete generated waveform.')

In [ ]:
vocoder=SpeechT5HifiGan.from_pretrained('microsoft/speecht5_hifigan',revision=VOCODER_REV)
synthesiser=pipeline('text-to-speech',model=ARTST_ID,revision=ARTST_REV,vocoder=vocoder,device='cpu')
embeddings_dataset=load_dataset('herwoww/arabic_xvector_embeddings',revision=EMBEDDING_REV,split='validation')
speaker_embedding=torch.tensor(embeddings_dataset[105]['speaker_embeddings'],dtype=torch.float32).unsqueeze(0)
assert tuple(speaker_embedding.shape)==(1,512)
text = '\nبدأ الاحتلال الإسرائيلي لفلسطين في عام 1967 خلال حرب الأيام الستة عندما \nاحتلت إسرائيل الضفة الغربية وقطاع غزة والقدس الشرقية. أصبحت هذه المناطق، التي يعيش فيها العديد من الفلسطينيين، محورًا للصراع منذ ذلك الحين. \nيرى المجتمع الدولي عمومًا أن المستوطنات الإسرائيلية هناك غير قانونية. \nتستمر الجهود نحو السلام، حيث يسعى الفلسطينيون إلى الاستقلال ويسعى الإسرائيليون إلى الأمن. \nلا تزال القضية معقدة للغاية ومثيرة للجدل.\n'
arabic_text=text
torch.manual_seed(42)
speech=synthesiser(text,forward_params={'speaker_embeddings':speaker_embedding})
arabic_audio=np.asarray(speech['audio']).reshape(-1)
arabic_rate=speech['sampling_rate']
wav_path=work_dir/'speech.wav'
sf.write(wav_path,arabic_audio,samplerate=arabic_rate)
display(IPythonAudio(arabic_audio,rate=arabic_rate))
print('Arabic waveform:',arabic_audio.shape,'| sample rate:',arabic_rate,'| duration:',round(len(arabic_audio)/arabic_rate,3),'seconds')
print('Saved complete mono speech.wav as PCM16.')

In [ ]:
for waveform,rate in [(english_audio,english_rate),(arabic_audio,arabic_rate)]:
    assert waveform.ndim==1 and len(waveform)>rate
    assert np.isfinite(waveform).all() and waveform.std()>1e-4
    assert np.max(np.abs(waveform))<=1.0
restored,rate=sf.read(wav_path,dtype='float32')
assert rate==arabic_rate and len(restored)==len(arabic_audio)
np.testing.assert_allclose(restored,np.clip(arabic_audio,-1,1-1/32768),atol=1/32768)
# Separate direct-model paths verify the pipelines' waveform handling.
english_inputs=narrator.tokenizer(english_text,return_tensors='pt')
torch.manual_seed(42)
with torch.inference_mode():
    direct_english=narrator.model(**english_inputs).waveform.cpu().numpy().reshape(-1)
np.testing.assert_allclose(english_audio,direct_english,rtol=1e-5,atol=1e-6)
arabic_inputs=synthesiser.tokenizer(arabic_text,return_tensors='pt')
assert arabic_inputs.input_ids.shape[1]<synthesiser.model.config.max_text_positions
torch.manual_seed(42)
with torch.inference_mode():
    direct_arabic=synthesiser.model.generate_speech(arabic_inputs.input_ids,speaker_embedding,vocoder=vocoder).cpu().numpy().reshape(-1)
np.testing.assert_allclose(arabic_audio,direct_arabic,rtol=1e-5,atol=1e-6)
print('Complete English and Arabic waveforms are finite and nonconstant; direct model outputs match both pipelines.')
print('Speaker row105 shape:',tuple(speaker_embedding.shape),'| Arabic input tokens:',arabic_inputs.input_ids.shape[1])
print('WAV round-trip sample rate, frame count and PCM16 quantization passed.')
workspace.cleanup()
print('Owned temporary audio file cleaned up.')

## Original source code

### Original block 1

```text
!pip install transformers
    !pip install -U datasets
    !pip install timm
    !pip install timm
    !pip install inflect
    !pip install phonemizer
```

### Original block 2

```text
from transformers import pipeline
from datasets import load_dataset
from IPython.display import Audio as IPythonAudio
import soundfile as sf
import torch
```

### Original block 3

```text
narrator = pipeline("text-to-speech",
                    model="./models/kakao-enterprise/vits-ljs")

text = """
The Israeli occupation of Palestine began in 1967 
during the Six-Day War when Israel captured the West Bank, 
Gaza Strip, and East Jerusalem. 
These areas, home to many Palestinians, have since been a 
focal point of conflict. The international community generally views Israeli settlements there as illegal. 
Efforts towards peace continue, with Palestinians seeking 
independence and Israelis seeking security. 
The situation remains highly complex and contentious.
"""

narrated_text = narrator(text)
```

### Original block 4

```text
from IPython.display import Audio as IPythonAudio

IPythonAudio(narrated_text["audio"][0],
             rate=narrated_text["sampling_rate"])
```

### Original block 5

```text
synthesiser = pipeline("text-to-speech", "MBZUAI/speecht5_tts_clartts_ar")

embeddings_dataset = load_dataset("herwoww/arabic_xvector_embeddings", split="validation")
speaker_embedding = torch.tensor(embeddings_dataset[105]["speaker_embeddings"]).unsqueeze(0)
# You can replace this embedding with your own as well.
text = """
بدأ الاحتلال الإسرائيلي لفلسطين في عام 1967 خلال حرب الأيام الستة عندما 
احتلت إسرائيل الضفة الغربية وقطاع غزة والقدس الشرقية. أصبحت هذه المناطق، التي يعيش فيها العديد من الفلسطينيين، محورًا للصراع منذ ذلك الحين. 
يرى المجتمع الدولي عمومًا أن المستوطنات الإسرائيلية هناك غير قانونية. 
تستمر الجهود نحو السلام، حيث يسعى الفلسطينيون إلى الاستقلال ويسعى الإسرائيليون إلى الأمن. 
لا تزال القضية معقدة للغاية ومثيرة للجدل.
"""
speech = synthesiser(text, forward_params={"speaker_embeddings": speaker_embedding})
# ArTST is trained without diacritics.

sf.write("speech.wav", speech["audio"], samplerate=speech["sampling_rate"])
```